In [1]:
import warnings ; warnings.filterwarnings('ignore')

import gym
import numpy as np

import random
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)
np.set_printoptions(suppress=True)
random.seed(123); np.random.seed(123)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
pip install git+https://github.com/mimoralea/gym-walk#egg=gym-walk

  Cloning https://github.com/mimoralea/gym-walk to /tmp/pip-install-xfbhxd2n/gym-walk_865719a36ae64b65b482cd230f7ae5e6
  Running command git clone --filter=blob:none --quiet https://github.com/mimoralea/gym-walk /tmp/pip-install-xfbhxd2n/gym-walk_865719a36ae64b65b482cd230f7ae5e6
  Resolved https://github.com/mimoralea/gym-walk to commit c453a5da6a37b6fadb393c490261bd6b9fa5bfc5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
def print_policy(pi, P, action_symbols=('<', 'v', '>', '^'), n_cols=4, title='Policy:'):
    print(title)
    arrs = {k:v for k,v in enumerate(action_symbols)}
    for s in range(len(P)):
        a = pi[s]
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), arrs[a].rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [4]:
def print_state_value_function(V, P, n_cols=4, prec=3, title='State-value function:'):
    print(title)
    for s in range(len(P)):
        v = V[s]
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), '{}'.format(np.round(v, prec)).rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [14]:
def probability_success(env, pi, goal_state, n_episodes=100, max_steps=200):
    random.seed(123); np.random.seed(123) ; env.seed(123)
    results = []
    for _ in range(n_episodes):
        state, done, steps = env.reset(), False, 0
        while not done and steps < max_steps:
            state, _, done, h = env.step(pi[state])
            steps += 1
        results.append(state == goal_state)
    return np.sum(results)/len(results)

In [15]:
def mean_return(env, pi, n_episodes=100, max_steps=200):
    random.seed(123); np.random.seed(123) ; env.seed(123)
    results = []
    for _ in range(n_episodes):
        state, done, steps = env.reset(), False, 0
        results.append(0.0)
        while not done and steps < max_steps:
            state, reward, done, _ = env.step(pi[state])
            results[-1] += reward
            steps += 1
    return np.mean(results)

In [16]:
env = gym.make('FrozenLake-v1')
P = env.env.P
init_state = env.reset()
goal_state = 15

In [17]:
P

{0: {0: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  2: [(0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)],
  3: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 1: {0: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False)],
  2: [(0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  3: [(0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 2:

Exponentially decaying schedule


In [18]:
def decay_schedule(
    init_value, min_value, decay_ratio,
    max_steps, log_start=-2, log_base=10):

    decay_steps = int(max_steps * decay_ratio)
    rem_steps = max_steps - decay_steps
    values = np.logspace(
        log_start, 0, decay_steps,
        base=log_base, endpoint=True
    )[::-1]
    values = (values - values.min()) / (values.max() - values.min())
    values = (init_value - min_value) * values + min_value
    values = np.pad(values, (0, rem_steps), 'edge')

    return values

Exploratory Policy Trajectories

In [19]:
from itertools import count
def generate_trajectory(
    select_action, Q, epsilon,
    env, max_steps=200):

    trajectory = []
    state = env.reset()
    for t in count():
        action = select_action(state, Q, epsilon)
        next_state, reward, done, _ = env.step(action)
        experience = (
            state,
            action,
            reward,
            next_state,
            done)
        trajectory.append(experience)
        if done:
            break
        if t >= max_steps - 1:
            break
        state = next_state
    return np.array(trajectory, object)

Monte Carlo control

In [20]:
from tqdm import tqdm

def mc_control(env, gamma=1.0,
               init_alpha=0.5, min_alpha=0.01, alpha_decay_ratio=0.5,
               init_epsilon=1.0, min_epsilon=0.1, epsilon_decay_ratio=0.9,
               n_episodes=3000, max_steps=200, first_visit=True):

    nS, nA = env.observation_space.n, env.action_space.n
    alphas = decay_schedule(
        init_alpha,
        min_alpha,
        alpha_decay_ratio,
        n_episodes
    )
    discounts = np.logspace(
        0, max_steps,
        num=max_steps,
        base=gamma,
        endpoint=False
    )
    epsilons = decay_schedule(
        init_epsilon,
        min_epsilon,
        epsilon_decay_ratio,
        n_episodes
    )
    pi_track = []
    Q = np.zeros((nS, nA), dtype=np.float64)
    Q_track = np.zeros((n_episodes, nS, nA), dtype=np.float64)

    select_action = lambda state, Q, epsilon: (
        np.argmax(Q[state])
        if np.random.random() > epsilon
        else np.random.randint(len(Q[state]))
    )
    for e in tqdm(range(n_episodes), leave=False):

        trajectory = generate_trajectory(
            select_action,
            Q,
            epsilons[e],
            env,
            max_steps
        )

        visited = np.zeros((nS, nA), dtype=bool)
        for t, (state, action, reward, _, _) in enumerate(trajectory):
            if visited[state][action] and first_visit:
                continue
            visited[state][action] = True
            n_steps = len(trajectory[t:])
            G = np.sum(discounts[:n_steps] * trajectory[t:, 2])
            Q[state][action] = Q[state][action] + \
                alphas[e] * (G - Q[state][action])
        Q_track[e] = Q
        pi_track.append(np.argmax(Q, axis=1))
    V = np.max(Q, axis=1)
    pi = np.argmax(Q, axis=1)

    return Q, V, pi

In [23]:
optimal_Q, optimal_V, optimal_pi = mc_control (env,n_episodes = 2999)

print('\nName:Easwari M')
print('Register Number:212223240033')
print_state_value_function(optimal_Q, P, n_cols=4, prec=2, title='Action-value function:')
print_state_value_function(optimal_V, P, n_cols=4, prec=2, title='State-value function:')
print_policy(optimal_pi, P)


Name:Easwari M
Register Number:212223240033
Action-value function:
| 00 [0.19 0.14 0.18 0.14] | 01 [0.03 0.06 0.07 0.15] | 02 [0.13 0.07 0.07 0.07] | 03 [0.02 0.01 0.   0.01] |
| 04 [0.19 0.12 0.1  0.11] |           | 06 [0.03 0.07 0.1  0.  ] |           |
| 08 [0.08 0.09 0.08 0.24] | 09 [0.17 0.2  0.26 0.1 ] | 10 [0.25 0.31 0.21 0.02] |           |
|           | 13 [0.08 0.06 0.38 0.15] | 14 [0.26 0.26 0.64 0.42] |           |
State-value function:
| 00   0.19 | 01   0.15 | 02   0.13 | 03   0.02 |
| 04   0.19 |           | 06    0.1 |           |
| 08   0.24 | 09   0.26 | 10   0.31 |           |
|           | 13   0.38 | 14   0.64 |           |
Policy:
| 00      < | 01      ^ | 02      < | 03      < |
| 04      < |           | 06      > |           |
| 08      ^ | 09      > | 10      v |           |
|           | 13      > | 14      > |           |


In [24]:
print('Name:Easwari M')
print('Register Number:212223240033')
print('Reaches goal {:.2f}%. Obtains an average undiscounted return of {:.4f}.'.format(
    probability_success(env, optimal_pi, goal_state=goal_state)*100,
    mean_return(env, optimal_pi)))


Name:Easwari M
Register Number:212223240033
Reaches goal 25.00%. Obtains an average undiscounted return of 0.2500.
